# 🔺 Delta Lake — Bloom Filters

---

## 1. O que é um Bloom Filter?
![Foto](./Fotos/18.png)
![Foto](./Fotos/19.png)
Um Bloom Filter é uma **estrutura binária probabilística** que permite ao Spark  
decidir rapidamente se um valor **definitivamente não está** em um arquivo — sem ler o arquivo.

Para funcionar, precisamos de três componentes:

```
  ┌──────────────────────────────────────────────────────────────┐
  │  1. ARQUIVO PARQUET                                          │
  │     Contém colunas de alta cardinalidade                     │
  │     ex: email, flight_id, user_id                            │
  │                                                              │
  │  2. ESTRUTURA BINÁRIA (o Bloom Filter em si)                 │
  │     Array de N bits, todos iniciando em 0                    │
  │     Uma estrutura por arquivo (não por tabela)               │
  │     ex: [ 0, 0, 0, 0, 0, 0, 0, 0, 0 ]  (9 posições)         │
  │           1  2  3  4  5  6  7  8  9                         │
  │                                                              │
  │  3. FUNÇÃO HASH                                              │
  │     Recebe um valor da coluna e retorna um número entre 1-N  │
  │     Aplicada a cada linha para construir o filtro            │
  └──────────────────────────────────────────────────────────────┘
```

---

### Como o Bloom Filter é construído

Para cada valor na coluna indexada, aplicamos a função hash e ativamos o bit correspondente:

```
  Coluna email no arquivo:
  ┌─────────────────────┐
  │ smith@gmail.com     │  hash → 3  →  bit[3] = 1
  │ johnson@gmail.com   │  hash → 8  →  bit[8] = 1
  │ williams@gmail.com  │  hash → 2  →  bit[2] = 1
  └─────────────────────┘

  Bloom Filter resultante:
  [ 0, 1, 1, 0, 0, 0, 0, 1, 0 ]
    1  2  3  4  5  6  7  8  9
```

---

### Como o Bloom Filter é usado em uma query

Na hora de executar `WHERE email = 'algum_valor'`, o Spark consulta o Bloom Filter antes de abrir o arquivo:

```
  Caso 1 — AUSÊNCIA GARANTIDA (nunca falso negativo):

  WHERE email = 'spark@gmail.com'
      │
      ▼
  hash('spark@gmail.com') = 5
      │
      ▼
  Bloom Filter: bit[5] = 0
      │
      ▼
  ✅ 100% garantido: 'spark@gmail.com' NÃO está neste arquivo → SKIP

  ─────────────────────────────────────────────────────────────────

  Caso 2 — PRESENÇA PROVÁVEL (possível falso positivo):

  WHERE email = 'kaminsky@gmail.com'
      │
      ▼
  hash('kaminsky@gmail.com') = 8
      │
      ▼
  Bloom Filter: bit[8] = 1
      │
      ▼
  ⚠️ Provavelmente está no arquivo → LER
  (pode ser falso positivo: outro valor gerou hash=8 antes)

  ─────────────────────────────────────────────────────────────────

  Caso 3 — PRESENÇA CONFIRMADA:

  WHERE email = 'smith@gmail.com'
      │
      ▼
  hash('smith@gmail.com') = 3
      │
      ▼
  Bloom Filter: bit[3] = 1
      │
      ▼
  ⚠️ Provavelmente está → LER → encontrado ✓
```

> **Garantia fundamental do Bloom Filter:**  
> - **Falso negativo: NUNCA** — se bit = 0, o valor definitivamente não está no arquivo  
> - **Falso positivo: POSSÍVEL** — se bit = 1, o valor provavelmente está (mas pode não estar)  
>  
> O Bloom Filter é excelente para **eliminar arquivos** — não para confirmá-los.

---

### Por que não usar o _delta_log para isso?

O `_delta_log` já guarda `minValues` e `maxValues` por coluna — mas **só para as primeiras 32 colunas**,  
e apenas para tipos com ordenação natural (INT, DATE, STRING básico).

```
  Coluna email (alta cardinalidade, posição 35 no schema):

  _delta_log stats:  min='alice@...', max='zara@...'  →  inútil para filtrar
                     (99% dos emails cabem nesse range)

  Bloom Filter:      hash lookup em O(1)  →  elimina arquivos com precisão
                     funciona para QUALQUER posição no schema
                     funciona para QUALQUER cardinalidade
```

---

### Estrutura de armazenamento no Delta Lake

```
  demo/
  ├── _delta_log/            ← commits e estatísticas por coluna
  ├── part-001.parquet       ← dados
  ├── part-002.parquet       ← dados
  └── _delta_index/          ← Bloom Filters (1 por arquivo Parquet)
        ├── part-001...index.v1.parquet
        └── part-002...index.v1.parquet
```

> Cada arquivo Parquet tem seu próprio Bloom Filter no `_delta_index/`.  
> O Spark consulta o filtro do arquivo antes de decidir se abre o Parquet.

---

## 2. Setup

In [ ]:
%python
# Desativa cache para garantir leituras reais do storage em cada query
spark.conf.set("spark.databricks.io.cache.enabled", False)

In [ ]:
%%sql
-- Exploração inicial do dataset
SELECT *
FROM CSV.`dbfs:/databricks-datasets/asa/airlines/2007.csv`
LIMIT 10;

---
## 3. Criando a tabela com Bloom Filter

Adicionamos a coluna `flight_id` — uma chave de alta cardinalidade que combina  
ano, mês, dia, companhia aérea e número do voo em uma string única.  
Essa coluna é o candidato ideal para o Bloom Filter:
- Alta cardinalidade (milhões de valores únicos)
- Usada em buscas pontuais (`WHERE flight_id = '...'`)
- Não seria eficientemente coberta pelas stats do `_delta_log`

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_bf;
CREATE TABLE flight_data_bf (
    Year INT, Month INT, DayOfMonth INT, DayOfWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING, flight_id STRING
);

In [ ]:
%%sql
-- Cria o índice Bloom Filter para a coluna flight_id
-- A partir daqui, cada arquivo Parquet terá um .index correspondente em _delta_index/
CREATE BLOOMFILTER INDEX
ON TABLE flight_data_bf
FOR COLUMNS(flight_id);

---
## 4. Inserindo os dados

Geramos a coluna `flight_id` como concatenação de `Year/Month/DayOfMonth-UniqueCarrier+FlightNum`  
ex: `2007/2/7-AA1863`

In [ ]:
%python
import pyspark.sql.functions as F

spark.read.csv("dbfs:/databricks-datasets/asa/airlines/{2007,2008}.csv", header=True, inferSchema=True)\
.withColumn("flight_id", F.concat(F.col("Year"), F.lit("/"), F.col("Month"), F.lit("/"), F.col("DayOfMonth"), \
    F.lit("-"), F.col("UniqueCarrier"), F.col("FlightNum"))) \
.write.mode("append").saveAsTable("flight_data_bf")

---
## 5. Inspecionando o Bloom Filter

### 5.1 Verificando a estrutura de diretórios

In [ ]:
-- Novo subdiretório _delta_index/ ao lado dos Parquets e do _delta_log
%fs
ls dbfs:/user/hive/warehouse/flight_data_bf

In [ ]:
-- Um arquivo .index por arquivo Parquet — cada um contém o Bloom Filter daquele arquivo
%fs
ls dbfs:/user/hive/warehouse/flight_data_bf/_delta_index/

### 5.2 Lendo o conteúdo do Bloom Filter

O Delta Lake **não permite** leitura direta dos arquivos `.index` via SQL — retorna erro.  
Usamos um workaround: copiamos o arquivo para um caminho acessível e lemos de lá.

In [ ]:
%%sql
-- ❌ Retorna erro — Delta Lake bloqueia leitura direta de arquivos .index
-- (substitua o path pelo nome real do arquivo listado acima)
SELECT * FROM PARQUET.`dbfs:/user/hive/warehouse/flight_data_bf/_delta_index/part-00000-f993607c-accf-46e1-b594-d48d3fe53b7a-c000.snappy.parquet.index.v1.parquet`

In [ ]:
%python
# Workaround: copia o arquivo de índice para um path acessível
# Substitua o nome do arquivo pelo listado no ls do _delta_index/
dbutils.fs.cp(
    "dbfs:/user/hive/warehouse/flight_data_bf/_delta_index/part-00000-f993607c-accf-46e1-b594-d48d3fe53b7a-c000.snappy.parquet.index.v1.parquet",
    "dbfs:/FileStore/index.parquet"
)

In [ ]:
%%sql
-- Leitura do Bloom Filter após a cópia — retorna os dados decodificados do índice
SELECT * 
FROM PARQUET.`dbfs:/FileStore/index.parquet`

### 5.3 Decodificando o Bloom Filter para binário

O valor retornado está em Base64. Decodificamos para ver a representação binária do filtro —  
cada bit `1` indica uma posição ativada por algum `flight_id` presente no arquivo.

In [ ]:
%python
import base64

encoded_bloom_filter = "AAAAQAAAAMAAASSEE+5P0Rw5nPleSZuoCfQmg6r9u623c5bhhxq7k114UPyJXUWpM6LpgH10eOUJNnAUG5z6Tpk1b1Q4Pp19ES75h0pIpxfjksYBEdrFLkreYUfi1RsqeGvCB6k/8reAGXRyUYvRunuZjfeF/BOKVvYAQ4Rganc="

decoded_bloom_filter = base64.b64decode(encoded_bloom_filter)
binary_representation = ''.join(f'{byte:08b}' for byte in decoded_bloom_filter)

# Exibe os primeiros 128 bits para visualização
print("Bloom Filter (primeiros 128 bits):")
print(binary_representation[:128])
print(f"\nTotal de bits: {len(binary_representation)}")
print(f"Bits ativos (1): {binary_representation.count('1')}")
print(f"Bits inativos (0): {binary_representation.count('0')}")

> **Interpretando o resultado:**  
> Cada `1` indica uma posição ativada pelo hash de algum `flight_id` presente no arquivo.  
> Cada `0` é uma posição que NENHUM valor do arquivo ativou —  
> qualquer `flight_id` cujo hash caia em uma posição `0` **definitivamente não está** neste arquivo.

---

## 6. Observando o Bloom Filter no plano de execução

`EXPLAIN EXTENDED` mostra o plano físico de execução da query, incluindo  
quais otimizações o Spark aplicou — entre elas, o uso do Bloom Filter.

In [ ]:
%%sql
EXPLAIN EXTENDED SELECT *
FROM flight_data_bf
WHERE flight_id IN ('2007/2/7-AA1863');

> **O que procurar no Physical Plan:**  
> Procure pela linha: `FileScan parquet ... with bloom filter ...`  
> Essa linha confirma que o Spark está usando o Bloom Filter para este scan.
>
> **Ponto importante sobre o exemplo:**  
> `flight_id` foi adicionada como última coluna — está **além das primeiras 32**.  
> O `_delta_log` não guarda `minValues`/`maxValues` para ela.  
> Sem o Bloom Filter, o Spark teria que **ler todos os arquivos** para encontrar o voo buscado.  
> Com o Bloom Filter, elimina arquivos em O(1) — mesmo para colunas fora das 32.

---

## 7. Verificando em qual coluna o Bloom Filter está definido

O `DESCRIBE DETAIL` não mostra essa informação. A forma correta é inspecionar  
os metadados do schema da tabela via PySpark:

In [ ]:
%python
# Exibe os metadados de cada coluna — colunas com Bloom Filter terão
# 'delta.bloomFilter' presente nos metadados
for field in spark.table("flight_data_bf").schema.fields:
    print(f"{field.name}: metadata={field.metadata}")

> Colunas com Bloom Filter terão `delta.bloomFilter` nos metadados com configurações  
> como `fpp` (false positive probability) e `numItems` (número estimado de itens únicos).

---

## 8. Resumo — Bloom Filters

```
┌──────────────────────────────────────────────────────────────────┐
│              BLOOM FILTER — FLUXO DE EXECUÇÃO                    │
│                                                                  │
│  WHERE flight_id = 'X'                                           │
│        │                                                         │
│        ▼                                                         │
│  Para cada arquivo Parquet:                                      │
│    1. Calcular hash('X')  →  posição P                           │
│    2. Consultar _delta_index/arquivo.index                       │
│              │                                                   │
│         bit[P] = 0  →  ✅ SKIP (100% garantido, sem ler Parquet) │
│         bit[P] = 1  →  ⚠️  LER  (provavelmente tem, pode ser FP) │
│                                                                  │
│  GARANTIAS:                                                      │
│  ✅ Falso Negativo: NUNCA  (bit=0 → valor ausente com certeza)    │
│  ⚠️  Falso Positivo: POSSÍVEL (bit=1 → pode estar ou não)        │
└──────────────────────────────────────────────────────────────────┘
```

### Bloom Filter vs outras estratégias de skip

| Estratégia | Funciona para colunas além da 32ª? | Efetivo para alta cardinalidade? | Automático? |
|---|---|---|---|
| **File Skipping** (`_delta_log` stats) | ❌ Apenas primeiras 32 | ❌ Min/max inútil para emails, IDs | ✅ |
| **Partitioning** | ❌ | ❌ Over-partitioning | ❌ |
| **Z-Ordering** | ❌ Apenas primeiras 32 | ✅ | ❌ |
| **Bloom Filter** | ✅ Qualquer posição | ✅ Ideal para alta cardinalidade | ❌ |

### Quando usar Bloom Filters

```
  ✅ Coluna com alta cardinalidade (email, ID, código de voo, hash)
  ✅ Coluna fora das primeiras 32 (onde _delta_log não guarda stats)
  ✅ Queries de busca pontual: WHERE col = 'valor' ou IN (...)
  ✅ Colunas usadas frequentemente em JOINs

  ❌ Colunas de baixa cardinalidade (Year, Country) → use Partitioning
  ❌ Range queries (WHERE id BETWEEN 1 AND 1000) → Bloom Filter não ajuda
  ❌ Tabelas muito pequenas → overhead não compensa
```

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Criar Bloom Filter | `CREATE BLOOMFILTER INDEX ON TABLE t FOR COLUMNS(col)` |
| Ver colunas com Bloom Filter | Loop em `spark.table('t').schema.fields` → `field.metadata` |
| Confirmar uso no plano | `EXPLAIN EXTENDED SELECT ... WHERE col = '...'` |
| Localizar arquivos de índice | `%fs ls dbfs:/.../_delta_index/` |